In [ ]:
from fastapi import APIRouter, HTTPException, status, Depends
from fastapi.security import HTTPBearer, HTTPAuthorizationCredentials

from app.schemas.auth import (
    RegisterRequest,
    LoginRequest,
    UserResponse,
    TokenResponse,
)
from app.core.security import (
    hash_password,
    verify_password,
    create_access_token,
    decode_access_token,
    access_token_ttl_seconds,
    TokenExpiredError,
)
from app.utils.helpers import generate_id, utc_now


router = APIRouter(
    prefix="/auth",
    tags=["Authentication"],
)


# auto_error=False so a missing/empty Authorization header produces a
# clear 401 from get_current_user (FastAPI's default is a generic
# 403 "Not authenticated", which the UI cannot tell apart from a
# permissions problem).
security = HTTPBearer(auto_error=False)


def get_database():
    """
    Return the MongoDB database attached to the FastAPI application.

    The database is initialized by notebooks/run_backend.ipynb.
    """
    from app.main import app

    database = getattr(app.state, "database", None)

    if database is None:
        raise RuntimeError("Database is not initialized.")

    return database


@router.post(
    "/register",
    response_model=TokenResponse,
    status_code=status.HTTP_201_CREATED,
)
def register(request: RegisterRequest):
    """
    Register a new learner.

    Flow:
    1. Validate RegisterRequest through Pydantic.
    2. Check whether the email already exists.
    3. Hash the password.
    4. Create the user in MongoDB.
    5. Generate a JWT.
    6. Return the token and user.
    """

    database = get_database()
    users = database.collection("users")

    # Normalize email so registration/login are case-insensitive.
    email = str(request.email).lower().strip()

    # Check for an existing account.
    existing_user = users.find_one({"email": email})

    if existing_user is not None:
        raise HTTPException(
            status_code=status.HTTP_409_CONFLICT,
            detail="An account with this email already exists.",
        )

    # Create the user document.
    user_id = generate_id()

    user_document = {
        "id": user_id,
        "email": email,
        "name": request.name.strip(),
        "password_hash": hash_password(request.password),
        "role": "student",
        "created_at": utc_now(),
        "updated_at": utc_now(),
    }

    try:
        users.insert_one(user_document)
    except Exception as exc:
        # MongoDB has a unique index on email.
        # This protects against a race condition where two registrations
        # use the same email simultaneously.
        if "duplicate key" in str(exc).lower():
            raise HTTPException(
                status_code=status.HTTP_409_CONFLICT,
                detail="An account with this email already exists.",
            )

        raise

    user = UserResponse(
        id=user_id,
        email=email,
        name=user_document["name"],
        role=user_document["role"],
    )

    access_token = create_access_token(
        {
            "sub": user_id,
            "email": email,
            "role": user_document["role"],
        }
    )

    return TokenResponse(
        access_token=access_token,
        token_type="bearer",
        expires_in=access_token_ttl_seconds(),
        user=user,
    )


@router.post(
    "/login",
    response_model=TokenResponse,
)
def login(request: LoginRequest):
    """
    Authenticate a learner and return an access token.
    """

    database = get_database()
    users = database.collection("users")

    email = str(request.email).lower().strip()

    user_document = users.find_one({"email": email})

    if user_document is None:
        raise HTTPException(
            status_code=status.HTTP_401_UNAUTHORIZED,
            detail="Invalid email or password.",
        )

    if not verify_password(
        request.password,
        user_document["password_hash"],
    ):
        raise HTTPException(
            status_code=status.HTTP_401_UNAUTHORIZED,
            detail="Invalid email or password.",
        )

    user = UserResponse(
        id=user_document["id"],
        email=user_document["email"],
        name=user_document["name"],
        role=user_document.get("role", "student"),
    )

    access_token = create_access_token(
        {
            "sub": user.id,
            "email": user.email,
            "role": user.role,
        }
    )

    return TokenResponse(
        access_token=access_token,
        token_type="bearer",
        expires_in=access_token_ttl_seconds(),
        user=user,
    )


def get_current_user(
    credentials: HTTPAuthorizationCredentials | None = Depends(security),
):
    """
    Resolve the authenticated user from the Authorization header.

    Every failure is a 401 with a specific reason so that a logout can
    be diagnosed (missing header vs expired vs bad signature) instead
    of all looking like the same generic error.
    """

    if credentials is None or not credentials.credentials:
        raise HTTPException(
            status_code=status.HTTP_401_UNAUTHORIZED,
            detail="Authentication token is missing.",
            headers={"WWW-Authenticate": "Bearer"},
        )

    token = credentials.credentials

    try:
        payload = decode_access_token(token)
    except TokenExpiredError:
        raise HTTPException(
            status_code=status.HTTP_401_UNAUTHORIZED,
            detail="Your session has expired. Please sign in again.",
            headers={"WWW-Authenticate": "Bearer"},
        )
    except Exception:
        raise HTTPException(
            status_code=status.HTTP_401_UNAUTHORIZED,
            detail="Invalid authentication token.",
            headers={"WWW-Authenticate": "Bearer"},
        )

    user_id = payload.get("sub")

    if not user_id:
        raise HTTPException(
            status_code=status.HTTP_401_UNAUTHORIZED,
            detail="Invalid authentication token.",
            headers={"WWW-Authenticate": "Bearer"},
        )

    database = get_database()
    users = database.collection("users")

    user_document = users.find_one({"id": user_id})

    if user_document is None:
        raise HTTPException(
            status_code=status.HTTP_401_UNAUTHORIZED,
            detail="User no longer exists.",
            headers={"WWW-Authenticate": "Bearer"},
        )

    return UserResponse(
        id=user_document["id"],
        email=user_document["email"],
        name=user_document["name"],
        role=user_document.get("role", "student"),
    )


@router.get(
    "/me",
    response_model=UserResponse,
)
def read_me(
    current_user: UserResponse = Depends(get_current_user),
):
    """Return the currently authenticated user."""

    return current_user


@router.post(
    "/refresh",
    response_model=TokenResponse,
)
def refresh_token(
    current_user: UserResponse = Depends(get_current_user),
):
    """
    Exchange a still-valid token for a fresh one (sliding session).

    The frontend calls this before the token expires, so an active
    user is never signed out mid-session.
    """

    access_token = create_access_token(
        {
            "sub": current_user.id,
            "email": current_user.email,
            "role": current_user.role,
        }
    )

    return TokenResponse(
        access_token=access_token,
        token_type="bearer",
        expires_in=access_token_ttl_seconds(),
        user=current_user,
    )
